# Assignment 1 – Feedforward Neural Network on MNIST

## Task 1 – Load the data
Same parameters and transformations as in the blog post (batch size 64 / 1000, normalisation with mean 0.1307 and std 0.3081).

In [ ]:
import time  # Zeitmessung (für den CPU/GPU-Vergleich)
import torch  # PyTorch-Grundbibliothek (Tensoren, Autograd)
import torchvision  # Datensätze und Bildtransformationen (u. a. MNIST)
import torch.nn as nn  # Bausteine für neuronale Netze (Layer, Module)
import torch.nn.functional as F  # Funktionen wie relu, log_softmax, nll_loss
import torch.optim as optim  # Optimierer (z. B. SGD)
import matplotlib.pyplot as plt  # Plotten der Ergebnisse

n_epochs = 50  # Anzahl der Durchläufe über den gesamten Trainingsdatensatz
batch_size_train = 64  # Anzahl Bilder pro Trainings-Batch
batch_size_test = 1000  # Anzahl Bilder pro Test-Batch
learning_rate = 0.01  # Schrittweite des Optimierers
momentum = 0.5  # Momentum-Faktor für SGD (glättet die Gradientenschritte)
log_interval = 10  # alle 10 Batches wird der Loss gespeichert
mean = 0.1307  # Mittelwert der MNIST-Pixel (für die Normalisierung)
standardDeviation = 0.3801  # Standardabweichung der MNIST-Pixel (für die Normalisierung)

random_seed = 1  # fester Startwert für reproduzierbare Ergebnisse
torch.backends.cudnn.enabled = False  # cuDNN aus -> deterministischere Ergebnisse auf der GPU
torch.manual_seed(random_seed)  # Zufallsgenerator von PyTorch initialisieren

train_loader = torch.utils.data.DataLoader(  # DataLoader liefert die Trainingsdaten in Batches
    torchvision.datasets.MNIST(  # MNIST-Datensatz laden
        "./data/",  # Speicherort der Daten
        train=True,  # Trainingsteil (60.000 Bilder)
        download=True,  # herunterladen, falls noch nicht vorhanden
        transform=torchvision.transforms.Compose(  # mehrere Transformationen hintereinander
            [
                torchvision.transforms.ToTensor(),  # Bild -> Tensor mit Werten in [0, 1]
                torchvision.transforms.Normalize((mean,), (standardDeviation,)),  # (x - mean) / std
            ]
        ),
    ),
    batch_size=batch_size_train,  # Batchgröße 64
    shuffle=True,  # Reihenfolge in jeder Epoche neu mischen
)

test_loader = torch.utils.data.DataLoader(  # DataLoader für die Testdaten
    torchvision.datasets.MNIST(  # MNIST-Datensatz laden
        "./data/",  # Speicherort der Daten
        train=False,  # Testteil (10.000 Bilder)
        download=True,  # herunterladen, falls noch nicht vorhanden
        transform=torchvision.transforms.Compose(  # gleiche Transformationen wie beim Training
            [
                torchvision.transforms.ToTensor(),  # Bild -> Tensor mit Werten in [0, 1]
                torchvision.transforms.Normalize((mean,), (standardDeviation,)),  # (x - mean) / std
            ]
        ),
    ),
    batch_size=batch_size_test,  # Batchgröße 1000
    shuffle=True,  # Testdaten ebenfalls mischen
)

examples = enumerate(test_loader)  # Iterator über die Test-Batches (mit Index)
batch_idx, (example_data, example_targets) = next(examples)  # ersten Test-Batch holen: Bilder + Labels

example_data.shape  # Form des Batches anzeigen: [1000, 1, 28, 28]

## Task 2 – Feedforward neural network
* Each image has shape `1×28×28`, so the input layer size is `D_in = 28*28 = 784` and the tensor is flattened in `forward`.
* ReLU in the hidden layer, no convolutions.
* `log_softmax` as output activation (pairs with the NLL loss).

In [ ]:
class TwoLayerNet(nn.Module):  # Netz mit einer versteckten Schicht, erbt von nn.Module
    def __init__(self, D_in, H, D_out):  # Konstruktor: Eingabe-, Hidden- und Ausgabegröße
        super(TwoLayerNet, self).__init__()  # Konstruktor der Basisklasse aufrufen
        self.linear1 = nn.Linear(D_in, H)  # vollverbundene Schicht: 784 -> 100
        self.linear2 = nn.Linear(H, D_out)  # vollverbundene Schicht: 100 -> 10

    def forward(self, x):  # Vorwärtsdurchlauf durch das Netz
        x = x.view(x.size(0), -1)  # flatten [N, 1, 28, 28] -> [N, 784]
        h_relu = F.relu(self.linear1(x))  # erste Schicht + ReLU-Aktivierung
        y_pred = self.linear2(h_relu)  # zweite Schicht liefert 10 Rohwerte (Logits)
        return F.log_softmax(y_pred, dim=-1)  # Log-Wahrscheinlichkeiten je Klasse (passt zu nll_loss)


# D_in is input dimension; H is hidden dimension; D_out is output dimension.
D_in, H, D_out = 28 * 28, 100, 10  # 784 Eingänge, 100 Hidden-Neuronen, 10 Klassen (Ziffern 0-9)

model = TwoLayerNet(D_in, H, D_out)  # Modell erzeugen
print(model)  # Aufbau des Modells ausgeben

## Task 3 – Train the network for 50 epochs
Negative log likelihood loss and SGD with the blog post's learning rate and momentum.
`train` and `test` take a `device` argument so the same code can run on CPU and GPU (Task 5).

In [ ]:
def train(model, optimizer, device, epoch, train_losses, train_counter):  # trainiert eine Epoche
    model.train()  # Trainingsmodus aktivieren (relevant z. B. für Dropout/BatchNorm)
    epoch_loss = 0  # Summe der Losses dieser Epoche

    for batch_idx, (data, target) in enumerate(train_loader):  # über alle Trainings-Batches iterieren
        data, target = data.to(device), target.to(device)  # Bilder und Labels auf CPU/GPU verschieben
        optimizer.zero_grad()  # Gradienten aus dem vorherigen Schritt löschen
        output = model(data)  # Vorwärtsdurchlauf: Vorhersagen berechnen
        loss = F.nll_loss(output, target)  # Negative-Log-Likelihood-Loss berechnen (Mittelwert über Batch)
        loss.backward()  # Backpropagation: Gradienten berechnen
        optimizer.step()  # Gewichte anhand der Gradienten aktualisieren
        epoch_loss += loss.item() * len(data)  # Loss mit Batchgröße gewichtet aufsummieren
        if batch_idx % log_interval == 0:  # nur jeden 10. Batch protokollieren
            train_losses.append(loss.item())  # aktuellen Loss speichern
            train_counter.append(  # Anzahl bisher gesehener Trainingsbilder speichern (x-Achse im Plot)
                (batch_idx * 64) + ((epoch - 1) * len(train_loader.dataset))
            )
    return epoch_loss / len(train_loader.dataset)  # durchschnittlicher Loss pro Bild in dieser Epoche


def test(model, device):  # bewertet das Modell auf den Testdaten
    model.eval()  # Evaluationsmodus aktivieren
    test_loss = 0  # Summe der Losses
    correct = 0  # Anzahl korrekter Vorhersagen
    with torch.no_grad():  # keine Gradienten berechnen (spart Speicher und Zeit)
        for data, target in test_loader:  # über alle Test-Batches iterieren
            data, target = data.to(device), target.to(device)  # Daten auf CPU/GPU verschieben
            output = model(data)  # Vorhersagen berechnen
            test_loss += F.nll_loss(output, target, reduction="sum").item()  # Loss aufsummieren (nicht mitteln)
            pred = output.argmax(dim=1)  # Klasse mit höchster Wahrscheinlichkeit = Vorhersage
            correct += pred.eq(target).sum().item()  # richtige Vorhersagen zählen
    test_loss /= len(test_loader.dataset)  # durchschnittlicher Loss pro Testbild
    accuracy = 100.0 * correct / len(test_loader.dataset)  # Genauigkeit in Prozent
    print(  # Ergebnis ausgeben
        "Test set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.2f}%)".format(
            test_loss, correct, len(test_loader.dataset), accuracy
        )
    )
    return test_loss, accuracy  # Loss und Genauigkeit zurückgeben


def run_training(device):  # komplettes Training über alle Epochen auf dem gegebenen Gerät
    torch.manual_seed(random_seed)  # same initial weights and shuffling for every run
    model = TwoLayerNet(D_in, H, D_out).to(device)  # neues Modell erzeugen und auf das Gerät verschieben
    optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)  # SGD-Optimierer mit Momentum
    train_losses, train_counter, epoch_losses = [], [], []  # Listen für die Loss-Verläufe

    start = time.perf_counter()  # Startzeit messen
    for epoch in range(1, n_epochs + 1):  # Epochen 1 bis 50
        epoch_losses.append(  # durchschnittlichen Epochen-Loss speichern
            train(model, optimizer, device, epoch, train_losses, train_counter)  # eine Epoche trainieren
        )
        if epoch == 1 or epoch % 5 == 0:  # nur Epoche 1 und jede 5. Epoche ausgeben
            print(  # Fortschritt ausgeben
                "Epoch {:2d}/{}  avg. train loss: {:.4f}".format(
                    epoch, n_epochs, epoch_losses[-1]
                )
            )
    if device.type == "cuda":  # nur auf der GPU nötig
        torch.cuda.synchronize()  # wait for all GPU work before stopping the clock
    duration = time.perf_counter() - start  # Trainingsdauer berechnen
    print("Training time on {}: {:.1f} s".format(device, duration))  # Trainingsdauer ausgeben
    return model, train_losses, train_counter, epoch_losses, duration  # Ergebnisse zurückgeben

In [ ]:
cpu = torch.device('cpu')  # CPU als Rechengerät festlegen
model_cpu, train_losses, train_counter, epoch_losses, time_cpu = run_training(cpu)  # Training auf der CPU starten

In [ ]:
fig = plt.figure(figsize=(9, 5))  # neue Abbildung (9x5 Zoll) anlegen
plt.plot(train_counter, train_losses, color='blue', alpha=0.4, linewidth=0.8, label='Train Loss (every 10th batch)')  # Loss jedes 10. Batches plotten
plt.plot([e * len(train_loader.dataset) for e in range(1, n_epochs + 1)], epoch_losses,  # x: gesehene Bilder am Ende jeder Epoche
         color='darkblue', marker='o', markersize=3, label='Train Loss (epoch average)')  # y: Epochen-Durchschnitt
plt.legend(loc='upper right')  # Legende oben rechts
plt.xlabel('number of training examples seen')  # Beschriftung x-Achse
plt.ylabel('negative log likelihood loss')  # Beschriftung y-Achse
plt.title('Training loss – TwoLayerNet on MNIST (50 epochs)')  # Titel
plt.ylim(0, 1)  # y-Achse auf 0 bis 1 begrenzen
plt.show()  # Plot anzeigen

## Task 4 – Test the network on the MNIST test data

In [ ]:
test_loss_cpu, accuracy_cpu = test(model_cpu, cpu)  # CPU-Modell auf den Testdaten auswerten

In [ ]:
with torch.no_grad():  # ohne Gradientenberechnung
  output = model_cpu(example_data)  # Vorhersagen für den Beispiel-Batch berechnen

fig = plt.figure()  # neue Abbildung anlegen
for i in range(6):  # die ersten 6 Bilder anzeigen
  plt.subplot(2,3,i+1)  # Raster mit 2 Zeilen x 3 Spalten, Position i+1
  plt.tight_layout()  # Abstände zwischen den Teilplots automatisch anpassen
  plt.imshow(example_data[i][0], cmap='gray', interpolation='none')  # Bild in Graustufen anzeigen
  plt.title("Prediction: {}".format(output.argmax(dim=1)[i].item()))  # vorhergesagte Ziffer als Titel
  plt.xticks([])  # x-Achsenmarkierungen ausblenden
  plt.yticks([])  # y-Achsenmarkierungen ausblenden
plt.show()  # Abbildung anzeigen

## Task 5 – Training on the GPU
PyTorch runs on the GPU when the **model** and the **data** live on a CUDA device:
1. Select the device: `device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')`
2. Move the model's parameters once: `model.to(device)` (before creating the optimizer)
3. Move every batch inside the training/test loop: `data, target = data.to(device), target.to(device)`

Because GPU calls are asynchronous, `torch.cuda.synchronize()` is called before the timer stops.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # GPU nutzen, falls vorhanden, sonst CPU
print('Using device:', device)  # gewähltes Gerät ausgeben
if device.type == 'cuda':  # falls GPU verwendet wird
  print(torch.cuda.get_device_name(0))  # Namen der GPU ausgeben

model_gpu, _, _, _, time_gpu = run_training(device)  # Training auf der GPU (nur Modell und Zeit werden behalten)
test_loss_gpu, accuracy_gpu = test(model_gpu, device)  # GPU-Modell auf den Testdaten auswerten

In [ ]:
print('{:<6} {:>12} {:>10} {:>10}'.format('Device', 'Train time', 'Test loss', 'Accuracy'))  # Tabellenkopf
print('{:<6} {:>10.1f} s {:>10.4f} {:>9.2f}%'.format('CPU', time_cpu, test_loss_cpu, accuracy_cpu))  # Zeile mit CPU-Ergebnissen
print('{:<6} {:>10.1f} s {:>10.4f} {:>9.2f}%'.format('GPU', time_gpu, test_loss_gpu, accuracy_gpu))  # Zeile mit GPU-Ergebnissen
print('Speed-up: {:.2f}x'.format(time_cpu / time_gpu))  # Beschleunigung GPU gegenüber CPU

**Discussion:**
* **Task 4:** After 50 epochs the network reaches **97.78 % accuracy** with an average test loss of **0.0718** on the 10,000 MNIST test images. The training loss (0.0108) is clearly lower than the test loss, i.e. the network starts to overfit slightly – the blog post's CNN with dropout does not have that gap.
* **Task 5:** CPU: 367.8 s, GPU (RTX 5060 Ti): 374.6 s → **no speed-up (0.98x)**. The network has only ~80k parameters and the batches are small (64), so the matrix multiplications are trivial for both devices. Training time is dominated by the `DataLoader` (reading every image, `ToTensor` and `Normalize` on the CPU for each batch) and the per-batch overhead of copying the data to the GPU and launching kernels. A GPU only pays off for larger models/batches (e.g. CNNs) or if the whole dataset is preloaded onto the GPU.
* With the same seed, CPU and GPU produce practically identical losses; the tiny differences in the last digits come from different floating-point implementations.